# Notebook 18b: Train Random Baseline Model

## Purpose
Train Random Baseline model (null hypothesis) on degree-binned data.

## Model
Random sampling from empirical distribution N(μ, σ²)
- No trainable parameters
- Predictions independent of features
- Serves as null hypothesis baseline

## Inputs
- results/pathway_nn/training_data/{metapath}_training_data.csv

## Outputs
- results/pathway_nn/trained_models/{metapath}_Random.pkl
- results/pathway_nn/benchmarks/{metapath}_Random_benchmark.json

In [ ]:
# Papermill parameters
metapath = 'CbGpPW'
record_benchmarks = True
random_seed = 42

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

repo_dir = Path.cwd().parent
sys.path.insert(0, str(repo_dir))

from src.models.random_baseline import RandomBaseline
from src.benchmarking import ModelBenchmarker

print(f"Training Random Baseline for {metapath}")

In [ ]:
# Create output directories
model_dir = repo_dir / 'results' / 'pathway_nn' / 'trained_models'
benchmark_dir = repo_dir / 'results' / 'pathway_nn' / 'benchmarks'
model_dir.mkdir(parents=True, exist_ok=True)
benchmark_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load training data
data_file = repo_dir / 'results' / 'pathway_nn' / 'training_data' / f'{metapath}_training_data.csv'
df = pd.read_csv(data_file)

print(f"Training data: {df.shape}")
print(f"  Pathway count mean range: [{df['pathway_count_mean'].min():.2f}, {df['pathway_count_mean'].max():.2f}]")

In [ ]:
# Prepare features and target
feature_cols = [c for c in df.columns if c.startswith('inter_sig_')]
X = df[['source_bin', 'target_bin'] + feature_cols].values
y = df['pathway_count_mean'].values

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")

In [ ]:
# Split train/test
n_train = int(0.8 * len(X))
indices = np.random.RandomState(random_seed).permutation(len(X))
train_idx = indices[:n_train]
test_idx = indices[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Train Model with Benchmarking

In [ ]:
# Initialize benchmarker
benchmarker = ModelBenchmarker(
    model_name='Random',
    metapath=metapath,
    n_training_samples=len(X_train),
    n_cores=1
)

# Train with timing
model = RandomBaseline(random_state=random_seed)

with benchmarker.time_training():
    model.fit(X_train, y_train)

print(f"Model trained: {model}")
print(f"Parameters: {model.get_params()}")

## Evaluate and Benchmark

In [ ]:
# Predict with timing
with benchmarker.time_prediction():
    predictions = model.predict(X_test)

# Record validation metrics
benchmarker.record_validation(predictions, y_test)

print(f"\nValidation Results:")
print(f"  Pearson r: {benchmarker.validation_r:.4f}")
print(f"  MAE: {benchmarker.validation_mae:.4f}")
print(f"  RMSE: {benchmarker.validation_rmse:.4f}")

## Save Model and Benchmark

In [ ]:
# Save model
model_file = model_dir / f'{metapath}_Random.pkl'
model.save(model_file)
print(f"✓ Saved model: {model_file}")

# Save benchmark
if record_benchmarks:
    result = benchmarker.finalize()
    benchmark_file = benchmark_dir / f'{metapath}_Random_benchmark.json'
    result.save_json(benchmark_file)
    print(f"✓ Saved benchmark: {benchmark_file}")
    print(f"\nBenchmark Summary:")
    print(f"  Training time: {result.training_time_sec:.2f}s")
    print(f"  Prediction time: {result.prediction_time_sec:.2f}s")
    print(f"  Peak memory: {result.peak_memory_gb:.2f} GB")
    print(f"  Normalized cost: {result.normalized_cost:.4f} node-hours")

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE: {metapath} Random Baseline")
print(f"{'='*70}")